# 第24课 · 一条乘法链撑起整个深度学习——链式法则（chain rule），反向传播的种子

**学习目标**：复合函数链式法则 $\frac{dy}{dx}=\frac{dy}{du}\cdot\frac{du}{dx}$——一层层连乘。先用 2–3 节点最小例子手推，这是反向传播的代数内核（L54–L56 装进 Value 引擎）。

**为什么对 Aurora 重要**：autograd 的 `backward()` 沿计算图连乘局部导数——本课公式就是那条链上的每一环。

← **上一课**　[L23 · 梯度](L23_gradients.ipynb)

> 上节课学习了 **梯度**：多元函数的"最陡上坡"方向，偏导与梯度向量的计算。  
> 本课将探讨 **链式法则**。

## 本课剧情：反向传播（backpropagation）靠什么？

神经网络有几十层。当你调整输入一点点，最终的损失会变多少？

答案是链式法则（chain rule）——函数套函数的求导规则：

```
y = f(g(x))  →  dy/dx = f'(g(x)) · g'(x)
```

直觉：内层把 x 的扰动放大（或缩小）g'(x) 倍，外层再把中间值的扰动放大 f'(g) 倍。两个放大倍数相乘，就是总的放大倍数。

**例子**：y = sin(x²)
- 外层 f(u) = sin(u)，f'(u) = cos(u)
- 内层 g(x) = x²，g'(x) = 2x
- dy/dx = cos(x²) · 2x

在 x=1 处：dy/dx = cos(1) · 2 ≈ 1.080

反向传播（backpropagation, BP）就是链式法则在计算图上的批量执行：  
从最后一层的误差出发，逐层向前"传递导数乘积"，直到每个参数的梯度都被算出来。

本课实现 `composite_derivative(x)` 并用数值差分验证，为 L56 反向传播铺路。

## 前置知识补充：本课涉及的三个重要概念

在深入链式法则之前，我们先把三个容易混淆的概念理清楚。

### 1. 为什么用中心差分？单侧差分不行吗？

高中数学中，导数定义是 $\lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$（单侧），这是从右侧逼近。  
在数值计算中，我们用中心差分 $\frac{f(x+h) - f(x-h)}{2h}$——从左右两侧同时逼近。

**为什么中心差分更精确？** 用泰勒展开：
- $f(x+h) = f(x) + f'(x)h + \frac{f''(x)}{2}h^2 + \frac{f'''(x)}{6}h^3 + O(h^4)$
- $f(x-h) = f(x) - f'(x)h + \frac{f''(x)}{2}h^2 - \frac{f'''(x)}{6}h^3 + O(h^4)$

两式相减：$f(x+h) - f(x-h) = 2f'(x)h + \frac{f'''(x)}{3}h^3 + O(h^5)$

因此 $\frac{f(x+h) - f(x-h)}{2h} = f'(x) + \frac{f'''(x)}{6}h^2 + O(h^4)$

关键：**误差项是 $O(h^2)$，而单侧差分的误差是 $O(h)$**。所以中心差分在步长 $h=10^{-5}$ 时误差约 $10^{-10}$，而单侧差分只有 $10^{-5}$。

### 2. 偏导符号 ∂ vs 导数符号 d 有什么区别？

在一元函数 $y=\sin(x^2)$ 中，为什么我们有时写 $\frac{dy}{dx}$，有时写 $\frac{\partial y}{\partial x}$？

- $\frac{dy}{dx}$："普通导数"——函数只有一个自变量，求导没有歧义
- $\frac{\partial y}{\partial x}$："偏导"——函数可能有多个自变量（如神经网络的多个权重），$\partial$ 强调"只改变 $x$，其他变量固定"

在一元函数里，两者完全等价。**我们在这里用 ∂ 是为了给你建立习惯——Month 2 的神经网络会有几百个权重，每个权重的"偏导"就是它的梯度。**

### 3. "局部导数"到底是什么？

在复合函数 $y=f(g(x))$ 中：

| 术语 | 含义 | 例子（y=sin(x²)） |
|---|---|---|
| **全局导数** | 从最终输出 $y$ 到原始输入 $x$ 的导数 | $\frac{dy}{dx} = \cos(x^2) \cdot 2x$ |
| **局部导数（边的导数）** | 计算图中某一条边上的导数（只看这条边，忽略其他部分） | 边 $x \to u$：$\frac{du}{dx}=2x$；边 $u \to y$：$\frac{dy}{du}=\cos(u)$ |

反向传播就是把"全局导数"拆成"局部导数"的乘积：$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$（这就是链式法则）。

---

## 1. 链式法则：分层剥开，逐层相乘

**为什么叫"链"？**

把复合函数写成一条链：

```
x  →  g(x)=x²  →  f(u)=sin(u)  →  y=sin(x²)
```

导数沿链传播，每一步乘以那一层的"局部斜率"：

$$\frac{dy}{dx} = \frac{df}{du} \cdot \frac{dg}{dx} = f'(g(x)) \cdot g'(x)$$

**手算步骤**（y = sin(x²)）：

| 步骤 | 含义 | 结果 |
|---|---|---|
| 拆层 | 令 u = g(x) = x²，y = f(u) = sin(u) | — |
| 内层导数 | du/dx = g'(x) = 2x | 2x |
| 外层导数 | dy/du = f'(u) = cos(u) = cos(x²) | cos(x²) |
| 链式乘积 | dy/dx = cos(x²) · 2x | cos(x²)·2x |

**数值验证（x=1.3）**：解析值 = cos(1.69)·2.6 ≈ -0.309；中心差分结果应在误差 1e-9 内。

> **n 层推广**：每加一层，乘法链就多一项。反向传播（backprop）正是把这条乘法链写成矩阵-向量乘积，批量传递梯度。

**注解：为什么表格中 f'(u) = cos(u) 最后变成了 cos(x²)？**

这是初学者最容易混淆的地方。我们分两个视角来看：

| 视角 | 过程 | 含义 |
|---|---|---|
| **关于 u 的导数** | $f(u) = \sin(u)$，所以 $\frac{df}{du} = \cos(u)$ | 这是外层函数的导数，它是 u 的函数 |
| **回到 x 的表达** | 因为 $u = x^2$，所以把 $u$ 换回 $x^2$：$f'(u) = \cos(u) = \cos(x^2)$ | 现在导数是 x 的函数了 |

换句话说：
- $f'(u)$ 是外层导数的"通用"形式——对任何 u 值都成立
- $f'(g(x)) = \cos(x^2)$ 是"特化"到我们这个复合函数的形式——现在它用 x 表示了

这两个式子不矛盾，只是同一个导数的不同写法。在代码中，我们会直接计算 `np.cos(x**2)`，这自动完成了这个代入。

---

### 严格推导：为什么是相乘而不是相加？

你的怀疑是对的——为什么不是 $\frac{dy}{dx} = \frac{dy}{du} + \frac{du}{dx}$？

用定义法从极限推导：

$$\frac{dy}{dx} = \lim_{h \to 0} \frac{f(g(x+h)) - f(g(x))}{h}$$

设 $\Delta u = g(x+h) - g(x)$，则 $g(x+h) = g(x) + \Delta u$。

$$\frac{dy}{dx} = \lim_{h \to 0} \frac{f(g(x) + \Delta u) - f(g(x))}{h}$$

分子分母同乘以 $\frac{\Delta u}{\Delta u}$（$\Delta u \ne 0$ 时）：

$$\frac{dy}{dx} = \lim_{h \to 0} \frac{f(g(x) + \Delta u) - f(g(x))}{\Delta u} \cdot \frac{\Delta u}{h}$$

第一项当 $h \to 0$ 时 $\Delta u \to 0$，所以 $\lim_{h \to 0} \frac{f(g(x) + \Delta u) - f(g(x))}{\Delta u} = f'(g(x))$；

第二项 $\lim_{h \to 0} \frac{\Delta u}{h} = \lim_{h \to 0} \frac{g(x+h) - g(x)}{h} = g'(x)$。

因此：$$\frac{dy}{dx} = f'(g(x)) \cdot g'(x)$$

**关键直觉**：内层的微小变化 $\Delta u$ 会被外层"放大"（如果 $f'(u)$ 很大）或"缩小"（如果 $f'(u)$ 很小）——这是乘法操作，不是加法。如果是加法，就意味着"无论内层怎么变，外层都是固定贡献"，这在数学上不成立。

---

## 实验入口：用数值变化观察函数

下面在 `x = 1.3` 处计算 `y = sin(x²)` 的数值导数，并与解析值 `cos(x²)·2x` 对比。关注 `numerical` 和 `analytical` 两列的误差大小。

In [ ]:
import numpy as np
x = 1.3
analytic = np.cos(x**2) * 2*x          # 链式法则
numeric = (np.sin((x+1e-5)**2) - np.sin((x-1e-5)**2)) / (2e-5)
print('链式法则:', round(analytic,5), ' 数值验证:', round(numeric,5))

> **[热身格 — 非链式法则]** 下面两格用最简单的多项式（`f(x)=x²`、`f(x)=x²+2x`）演练「数值差分」技术本身，
> 不涉及函数嵌套，因此**不是链式法则的应用**。目的是让你在接触复合函数前先确认差分公式的精度。
> 真正的链式法则示例从下一节（2. 实现 `composite_derivative`）开始。

## 动手观察：变化率不是一句口号

在一组 `x` 值上同时打印函数值和近似斜率，观察数值差分在每个点的精度。

In [ ]:
import numpy as np

# 实例：f(x) = sin(x)，在 x=1 处的导数是 cos(1) ≈ 0.5403
# 用两种差分公式分别近似，看误差怎么变化
f = lambda x: np.sin(x)
true_derivative = np.cos(1.0)

print("用不同步长 h 对比中心差分 vs 单侧差分的精度")
print(f"真实导数 cos(1) = {true_derivative:.8f}\n")
print(f"{'h':>10}  {'单侧差分误差':>16}  {'中心差分误差':>16}  {'精度提升倍数':>12}")
print("-" * 60)

for h_power in range(-3, -12, -1):  # h = 1e-3, 1e-4, ..., 1e-11
    h = 10.0 ** h_power
    
    # 单侧差分：[f(x+h) - f(x)] / h
    one_sided = (f(1.0 + h) - f(1.0)) / h
    err_one_sided = abs(one_sided - true_derivative)
    
    # 中心差分：[f(x+h) - f(x-h)] / (2h)
    center = (f(1.0 + h) - f(1.0 - h)) / (2 * h)
    err_center = abs(center - true_derivative)
    
    if err_center > 0:
        improvement = err_one_sided / err_center
    else:
        improvement = float('inf')
    
    print(f"1e{h_power:2d}  {err_one_sided:16.2e}  {err_center:16.2e}  {improvement:12.1f}x")
    
print("\n观察：")
print("- 随着 h 减小，单侧差分的误差线性下降（斜率 -1）")
print("- 随着 h 减小，中心差分的误差二次下降（斜率 -2）")
print("- 在 h=1e-7 处，中心差分的误差约 10e-15，已经接近浮点数精度上限")
print("- 这就是为什么代码实验中用 h=1e-5 或 1e-7：平衡精度与数值稳定性")

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('近似斜率 =', np.round(slopes, 3))
print('理论斜率 2x =', 2 * xs)


## 代码实验：遍历不同位置，看斜率如何变化

下面遍历一列 `x`，把函数值和解析导数并排输出，确认链式法则在每个点都与数值差分吻合。

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ 实现 `composite_derivative(x)` —— y=sin(x²) 的导数

**推理路线**：
1. 拆层：外层 f(u)=sin(u)，内层 g(x)=x²，链式法则给出 dy/dx = f'(g(x)) · g'(x)
2. 分别求导：f'(u)=cos(u)，代入 u=x² 得 cos(x²)；g'(x)=2x
3. 相乘：结果是 cos(x²)·2x，NumPy 广播自动保证对数组输入也正确

**参考输入输出**：x=√(π/2) 时，x²=π/2，dy/dx=cos(π/2)·2x=0·√(2π)=0（sin(x²) 在该点取极值，导数为零）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


## 旁白：为什么不直接用公式 dy/dx = cos(x²)·2x？

你可能会问：既然已经推导出了答案 dy/dx = cos(x²)·2x，为什么还要分"前向"和"反向"来算？

**在这个2层例子中，两种方法结果相同。** 但当函数变得复杂（如神经网络的几十层）时：

| 方法 | 复杂度 | 适用场景 |
|---|---|---|
| **直接法** | 需要用手（或符号计算工具）逐层推导完整公式 | 函数简单（≤ 3层）；理论推导 |
| **前向-反向法** | 逐层缓存局部导数，然后按逆序相乘 | 函数复杂（几十层）；**自动求导（autograd）的基础** |

在深度学习中，一个神经网络往往有 50+ 层，每层有数百万参数。如果要对每个参数求导，直接法会产生一个巨大的符号表达式（几页纸都写不完）。

**但前向-反向法只需要：**
1. 前向：按顺序计算每一层，同时记录该层的导数（几行代码）
2. 反向：从最后一层的损失开始，向前连乘这些导数（几行代码）

这个想法就是 Month 2 的 `autograd` 引擎的核心——机器会自动做"分拆-连乘"这个过程，我们不用手写任何导数公式。

下面的 `composite_derivative` 就是这个思想的最小实例：我们不写完整的公式 cos(x²)·2x，而是明确拆出"内层的2x"和"外层的cos(u)"，让代码清晰地表达"逐层求导，再相乘"的意图。

---

### 写代码前，先把变量表补完整

写 `composite_derivative` 前明确三件事：
- 输入：`x`（标量或数组，单位弧度）
- 关键步骤：先算内层 `g(x) = x²`，再把外层导数 `cos(x²)` 和内层导数 `2x` 相乘
- 返回：与 `x` 等形状的导数数组，可直接和数值差分对比

In [ ]:
def composite_derivative(x):
    # ✏️ TODO: sin(x^2) 对 x 的导数
    # 提示：外层 f(u)=sin(u) → f'(u)=cos(u)；内层 g(x)=x² → g'(x)=2x
    # 链式法则：dy/dx = cos(x²) · 2x
    raise NotImplementedError("TODO: 实现 composite_derivative — 返回 cos(x**2) * 2*x")


In [ ]:
x = 1.3
numeric = (np.sin((x+1e-5)**2) - np.sin((x-1e-5)**2)) / (2e-5)
try:
    result = composite_derivative(x)
    assert abs(result - numeric) < 1e-6, (
        f"误差 {abs(result - numeric):.2e} 过大（应 < 1e-6）"
    )
    print("✅ 通过：你手算了一次链式法则 = 一次最小的反向传播。")
except (NotImplementedError, TypeError):
    print("⚠️  composite_derivative 尚未实现，请先完成 TODO 再运行此格。")


In [ ]:
# Aurora 连接：Month 2 的 autograd 引擎（L54 的 Value.backward）就是把本课的链式乘法自动化。
# 手动模拟一次最小的 "backward"：y = sin(x²) 的计算图只有两条边，各自缓存局部导数。
import numpy as np

x = 1.3
u = x**2                  # 前向：内层节点 g(x)
y = np.sin(u)             # 前向：外层节点 f(u)
local_du_dx = 2 * x       # 边 x→u 的局部导数（前向时就能缓存）
local_dy_du = np.cos(u)   # 边 u→y 的局部导数（前向时就能缓存）
grad = local_dy_du * local_du_dx   # backward：沿计算图的边连乘
print(f'dy/dx = {grad:.6f}（与链式法则手算 cos(x²)·2x 一致）')
assert abs(grad - np.cos(x**2) * 2 * x) < 1e-12
print('✅ 这几行就是 autograd backward 的最小雏形——L54 会把它封装成 Value 类')

## 计算图长什么样？用代码显式表示

你看上面的代码，其实就是在构建一个"计算图"。让我们把这个图用数据结构显式写出来：

In [ ]:
import numpy as np

# y = sin(x²) 的计算图用字典表示
x = 1.3
computation_graph = {
    "nodes": {
        "input_x": {"value": x, "type": "input"},
        "u": {"value": x**2, "type": "intermediate"},
        "output_y": {"value": np.sin(x**2), "type": "output"},
    },
    "edges": [
        {
            "from": "input_x",
            "to": "u",
            "operation": "lambda x: x**2",
            "local_gradient": 2*x,  # du/dx at this x
        },
        {
            "from": "u",
            "to": "output_y",
            "operation": "lambda u: np.sin(u)",
            "local_gradient": np.cos(x**2),  # dy/du at this u
        }
    ]
}

print("计算图的节点：")
for node_name, node_info in computation_graph["nodes"].items():
    print(f"  {node_name}: value={node_info['value']:.6f}, type={node_info['type']}")

print("\n计算图的边（每条边都存储一个局部导数）：")
for i, edge in enumerate(computation_graph["edges"]):
    print(f"  边{i}: {edge['from']} -> {edge['to']}")
    print(f"      操作: {edge['operation']}")
    print(f"      局部导数: {edge['local_gradient']:.6f}")

print("\n反向传播：从输出层向输入层传递梯度")
print("  1. 起点：输出 y 的导数是 1（自己对自己的导数）")
print("  2. 沿边反向：dy/dx = dy/du * du/dx")
dy_du = computation_graph["edges"][1]["local_gradient"]
du_dx = computation_graph["edges"][0]["local_gradient"]
dy_dx = dy_du * du_dx
print(f"           = {dy_du:.6f} * {du_dx:.6f} = {dy_dx:.6f}")
print("\n💡 这个过程就是反向传播——把梯度沿计算图的边反向连乘。\n   Month 2 的 autograd 会把这个字典的构建和梯度传递自动化。")

## 参数实验：3 层复合的链式法则

构造 3 层复合 h(x)=sin(exp(x²))：
- 最内层 u=x²，导数 du/dx=2x
- 中间层 v=exp(u)，导数 dv/du=exp(u)=exp(x²)
- 外层 y=sin(v)，导数 dy/dv=cos(v)=cos(exp(x²))

链式法则给出：dh/dx = cos(exp(x²)) · exp(x²) · 2x

用步长 `h=1e-7` 的数值差分在 x=[0, 1, -1] 处各算一次数值导数，与上式手算值对比，误差应在 1e-9 以内。这验证了链式法则可直接推广到任意深度的嵌套——autograd 计算图做的正是同一件事。

## 链式法则推广到3层：h(x) = sin(exp(x²))

当复合函数有 3 层时，链式法则如何推广？规则不变——**从外向内一层层乘下去**。

让我们用表格明确每一步：

| 步骤 | 函数 | 变量 | 导数公式 | 在我们例子中 |
|---|---|---|---|---|
| **1. 最外层（从输出开始）** | $y = \sin(v)$ | $v$ 是中间值 | $\frac{dy}{dv} = \cos(v)$ | $\cos(\exp(x^2))$ |
| **2. 中间层** | $v = \exp(u)$ | $u$ 是更里层的中间值 | $\frac{dv}{du} = \exp(u)$ | $\exp(x^2)$ |
| **3. 最内层（到达输入）** | $u = x^2$ | $x$ 是最终输入 | $\frac{du}{dx} = 2x$ | $2x$ |
| **4. 链式乘积** | — | — | $\frac{dh}{dx} = \frac{dy}{dv} \cdot \frac{dv}{du} \cdot \frac{du}{dx}$ | $\cos(\exp(x^2)) \cdot \exp(x^2) \cdot 2x$ |

**记住顺序**：$\frac{dh}{dx} = [\text{最外层导数}] \times [\text{中层导数}] \times [\text{最内层导数}]$

**直觉**：x 的微小变化先被最内层（x²）的导数"放大"2x 倍，这个变化再被中层（exp）的导数放大 exp(x²) 倍，最后被外层（sin）的导数放大 cos(...) 倍。三个放大倍数连乘，就是总的放大倍数。

---

In [ ]:
# 参数实验：h(x) = sin(exp(x²)) 链式法则验证
# dh/dx = cos(exp(x²)) · exp(x²) · 2x
def h_analytic(x):
    return np.cos(np.exp(x**2)) * np.exp(x**2) * 2 * x

print(f"{'x':>6}  {'解析导数':>14}  {'数值差分':>14}  {'误差':>10}")
print('-' * 50)
for x in [0.0, 1.0, -1.0]:
    analytic = h_analytic(x)
    numerical = (np.sin(np.exp((x+1e-7)**2)) - np.sin(np.exp((x-1e-7)**2))) / 2e-7
    err = abs(analytic - numerical)
    print(f"{x:>6.1f}  {analytic:>14.8f}  {numerical:>14.8f}  {err:>10.2e}")
    assert err < 1e-9, f'x={x}: 误差 {err} 过大'
print('✅ sin(exp(x²)) 的链式法则在三点处均验证通过')


### 补充知识：指数函数 e^x 的导数

在上面的 3 层链式法则中，中层是 $v = \exp(u) = e^u$（$e \approx 2.718$ 是数学常数）。

**高中可能没学过，所以这里补充一下：**

指数函数 $y = e^x$ 的导数是它自己：

$$\frac{d}{dx} e^x = e^x$$

**为什么？** 这是 $e$ 的定义——它是唯一使得"函数的导数等于函数本身"的底数。如果用 $2^x$ 或 $10^x$，导数会不同（有个常数因子）。

在我们的例子中，当中层导数是 $\frac{dv}{du} = e^u$ 时，它在 $u=x^2$ 处的值就是 $e^{x^2}$。

**记住这个结论就够了**（完整推导需要极限知识，高中阶段不要求）。

---

In [ ]:
# 扩展：4 层复合 g(x) = tanh(sin(x²+1))
# dg/dx = (1 - tanh²(sin(x²+1))) · cos(x²+1) · 2x
def g_analytic(x):
    u = x**2 + 1
    v = np.sin(u)
    w = np.tanh(v)
    return (1 - w**2) * np.cos(u) * 2 * x

x = 0.8
analytic = g_analytic(x)
numerical = (np.tanh(np.sin((x+1e-7)**2+1)) - np.tanh(np.sin((x-1e-7)**2+1))) / 2e-7
err = abs(analytic - numerical)
print(f'x={x}: 解析导数={analytic:.8f}, 数值差分={numerical:.8f}, 误差={err:.2e}')
assert err < 1e-5
print('✅ 4 层复合链式法则验证通过')
print('  autograd 做的正是这件事——只是参数空间是百万维，链更深')


### 补充知识：双曲正切函数 tanh 的导数

在 4 层复合函数中出现了 $y = \tanh(x)$ 和它的导数 $\frac{dy}{dx} = 1 - \tanh^2(x)$。这也是高中没有学过的函数。

**tanh 是什么？** 双曲正切（hyperbolic tangent）的缩写。它的定义是：

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

虽然定义看起来复杂，但它的性质很好：
- 值域在 (-1, 1) 之间（像一个平滑的 S 形曲线）
- 导数公式简洁：$\frac{d}{dx} \tanh(x) = 1 - \tanh^2(x)$

**为什么 1 - tanh²(x)？** 这可以从上面的定义推导出来（用商法则和 e^x 的导数），但过程较复杂。在这里，你只需要记住这个公式。

在神经网络中，$\tanh$ 经常用作**激活函数**（Month 1 会讲）——它比 sigmoid 训练得更快。

**在本课中的用处**：这是一个 4 层复合函数的验证例子，目的是展示链式法则适用于任意复杂度。不需要理解 tanh 的完整推导，只需要相信导数公式，并用数值差分验证答案是否正确。

---

## 🎯 未来的回报 (Future Payoff)

今天你亲手拆开的这条「外层导数 × 内层导数」乘法链，是整个深度学习求导的种子。它会在 **L54（autograd）** 变成 `Value.backward()`，并在 **L56（反向传播 backpropagation）** 马上回来——那时整张神经网络的求导，本质上就是把今天这条链沿着计算图一层层连乘。

## 本课收束

现在可以调用 `composite_derivative(x)` 求出 `sin(x²)` 在任意点的导数，数值误差通常在 `1e-9` 量级内。这个“外层导数 × 内层导数”的结构，正是 `autograd.backward()` 的核心：每个节点缓存局部导数，再沿计算图向输入方向连乘。等 Month 2 真正实现 autograd 时，`composite_derivative` 这种拆法会直接变成计算图的边和节点设计。

下一课：**L25**（梯度下降，gradient descent）把今天得到的导数当方向信号，把参数沿负梯度方向推动一步。更远处的 **L54**（autograd）会把这条链式法则直接变成 `Value.backward()`，用计算图自动完成同样的乘法。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：链式法则手算（目标 8 分钟）

盖上屏幕，纸上作答：

**问 1**：y = sin(x²)，写出链式法则三步（拆层、内层导数、外层导数、乘积）。  
在 x=1 处，dy/dx = ?（算出数值）

**问 2**：y = (3x+1)⁵，dy/dx = ?  
（提示：外层 f(u)=u⁵，内层 g(x)=3x+1）

**问 3**：h(x) = sin(exp(x²))，写出 dh/dx 的链式法则展开。  
（3层复合，每层各乘一项）

**问 4**：反向传播中，"链式法则"的作用是什么？  
（一句话解释：从最终误差到第 i 层参数梯度的计算路径）

推导完成后运行下面格对答案。

### 答题指南

这个白板挑战涵盖了本课的所有关键思想：
1. **问 1-3**：逐个应用链式法则，层数从 2、3 递增
2. **问 4**：理解链式法则与反向传播的关系

做题时记住：
- 先拆层（内层函数是什么？外层函数是什么？）
- 分别求导（每层的导数是什么？）
- 从外向内相乘（不要搞反顺序）

### 快速复习：幂函数求导公式

在问 2 中会用到 $\frac{d}{dx} x^n = n \cdot x^{n-1}$。这是高中数学的基础公式，你可能需要快速回忆一下：

- $\frac{d}{dx} x = 1$
- $\frac{d}{dx} x^2 = 2x$
- $\frac{d}{dx} x^3 = 3x^2$
- $\frac{d}{dx} x^5 = 5x^4$（问 2 会用到这个）

**推导思路**（用二项式定理）：$(x+h)^n = x^n + nx^{n-1}h + O(h^2)$，所以 $\frac{(x+h)^n - x^n}{h} = nx^{n-1} + O(h)$，当 $h \to 0$ 时得 $nx^{n-1}$。

---

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：y=sin(x²)，x=1 处 dy/dx = cos(1)·2
x1 = 1.0
dy_dx1 = np.cos(x1**2) * 2 * x1
assert np.isclose(dy_dx1, 2 * np.cos(1.0), atol=1e-12)
try:
    cd1 = composite_derivative(x1)
    assert abs(cd1 - dy_dx1) < 1e-7, f"composite_derivative 与解析值不符: {cd1} vs {dy_dx1}"
    print(f"Q1 ✅  dy/dx|_{{x=1}} = cos(1)·2 = {dy_dx1:.6f}，数值={cd1:.6f}")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  dy/dx|_{{x=1}} = cos(1)·2 = {dy_dx1:.6f}  (composite_derivative 待实现)")

# 问2：y=(3x+1)⁵，dy/dx = 5(3x+1)⁴·3 = 15(3x+1)⁴
f2 = lambda x: (3*x + 1)**5
df2_analytical = lambda x: 15 * (3*x + 1)**4
x2 = 0.5
expected2 = df2_analytical(x2)
numeric2 = (f2(x2 + 1e-7) - f2(x2 - 1e-7)) / (2e-7)
assert np.isclose(numeric2, expected2, rtol=1e-5), f"(3x+1)⁵ 导数不符: {numeric2} vs {expected2}"
print(f"Q2 ✅  d/dx(3x+1)⁵ = 15(3x+1)⁴，在x={x2}处={expected2:.4f}，数值={numeric2:.4f}")

# 问3：h(x)=sin(exp(x²))，dh/dx = cos(exp(x²))·exp(x²)·2x
h = lambda x: np.sin(np.exp(x**2))
dh_analytical = lambda x: np.cos(np.exp(x**2)) * np.exp(x**2) * 2*x
x3 = 0.5
expected3 = dh_analytical(x3)
numeric3 = (h(x3 + 1e-7) - h(x3 - 1e-7)) / (2e-7)
assert np.isclose(numeric3, expected3, rtol=1e-5), f"sin(exp(x²)) 导数不符"
print(f"Q3 ✅  dh/dx=cos(e^x²)·e^x²·2x，在x={x3}处={expected3:.6f}，数值={numeric3:.6f}")

# 问4：反向传播是链式法则的批量执行（验证2层链式法则）
# y=f(g(x)), dy/dx=f'(g(x))·g'(x) — 验证乘法链
g = lambda x: x**2
f_outer = lambda u: np.sin(u)
dg = lambda x: 2*x
df = lambda u: np.cos(u)
x4 = 1.3
chain = df(g(x4)) * dg(x4)
direct_num = (f_outer(g(x4 + 1e-7)) - f_outer(g(x4 - 1e-7))) / (2e-7)
assert np.isclose(chain, direct_num, rtol=1e-6)
print(f"Q4 ✅  链式积 f'(g(x))·g'(x) = {chain:.6f} = 数值差分 {direct_num:.6f}")
print("     BP = 从输出层误差出发，逐层向前乘以该层局部导数，直到参数层")
print("\n🎉 链式法则白板挑战通过！反向传播的数学基础已内化。")

In [ ]:
# ✏️ 本课自评
l24_review = {
    "chain_rule_formula":         None,  # 记住 dy/dx = f'(g(x))·g'(x)？True/False
    "composite_deriv_done":       None,  # composite_derivative 实现并通过断言？True/False
    "3_layer_chain":              None,  # 能展开3层复合的链式乘积？True/False
    "backprop_connection":        None,  # 理解BP=链式法则在计算图上批量执行？True/False
    "whiteboard_passed":          None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l24_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l24_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L24 全部通关！进入 L25：梯度下降')

---

→ **下一课**　[L25 · 梯度下降（gradient descent）](L25_gradient_descent.ipynb)

> 下节课将学习 **梯度下降**：蒙眼下山、沿负梯度走到谷底，从损失函数到权重更新公式。